# MPM paper example replication
This notebook reproduces the finite-action masked-perturbation game example.

## Import setup
If `mpmgame` is not installed in the notebook kernel, this cell adds the repository `src/` directory to `sys.path`.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    src_dir = candidate / 'src'
    if (src_dir / 'mpmgame').exists():
        sys.path.insert(0, str(src_dir))
        break

import numpy as np
import mpmgame as mpm
from mpmgame.examples import coordinated_attack_defeats_all
print('Using mpmgame from:', mpm.__file__)


Using mpmgame from: C:\Users\perry\Documents\VSCode\masked-perturbation-model\src\mpmgame\__init__.py


## 1) Build the paper example data and admissible defenses
This block instantiates the paper transfer matrices/attacks/costs and computes admissible defenses under budget constraints.

**What it demonstrates:** the concrete finite action sets used in the game analysis.

**Expected behavior:** printed admissible masks should match the documented example set.

In [2]:
data = mpm.paper_example_data()
admissible = mpm.admissible_defenses(2, 2, data.c_w, data.c_r, data.budget, include_empty=False)
print('Admissible defenses from costs/budget:')
for m in admissible:
    print(m)


Admissible defenses from costs/budget:
[[0 0]
 [1 1]]
[[1 0]
 [0 0]]
[[1 1]
 [0 0]]
[[1 0]
 [1 0]]
[[1 1]
 [1 1]]


## 2) Compute success sets
This block computes attack-success and defense-success sets.

**Where it fits:** these sets drive dominance checks and reduced-game construction.

**Expected behavior:** printed sets should match the chapter/paper values used for acceptance checks.

In [3]:
success = mpm.compute_success_sets(data.M, data.attacks, data.defenses)
print('Attack success sets:')
for k, v in success.attack_success.items():
    print(k, sorted(v))
print('Defense success sets:')
for k, v in success.defense_success.items():
    print(k, sorted(v))


Attack success sets:
Δ1 ['∇1', '∇2', '∇4']
Δ2 ['∇3']
Δ3 ['∇2']
Defense success sets:
∇1 ['Δ2', 'Δ3']
∇2 ['Δ2']
∇3 ['Δ1', 'Δ3']
∇4 ['Δ2', 'Δ3']


## 3) Identify dominated actions and reduced game
This block computes dominated attacks/defenses and forms the reduced game.

**What it demonstrates:** strategic simplification before mixed-strategy solving.

**Expected behavior:** reduced labels and payoff should align with the worked example.

In [4]:
dom_a = mpm.dominated_attacks(success.attack_success)
dom_d = mpm.dominated_defenses(success.defense_success)
print('Dominated attacks:', sorted(dom_a))
print('Dominated defenses:', sorted(dom_d))
reduced = mpm.eliminate_dominated_strategies(data.M, data.attacks, data.defenses)
print('Reduced attacks:', [a.label for a in reduced.attacks])
print('Reduced defenses:', [d.label for d in reduced.defenses])
print('Reduced payoff:\n', reduced.payoff)


Dominated attacks: ['Δ3']
Dominated defenses: ['∇2']
Reduced attacks: ['Δ1', 'Δ2']
Reduced defenses: ['∇1', '∇3', '∇4']
Reduced payoff:
 [[1. 0. 1.]
 [0. 1. 0.]]


## 4) Solve mixed-strategy equilibrium
This block solves the reduced zero-sum game and reports optimal attacker/defender mixes and value.

**Expected behavior:** approximately 50/50 mixtures and value near 0.5 for the canonical reduced 2x2 example.

In [5]:
p, q, v = mpm.solve_zero_sum_game(reduced.payoff)
print('Attacker optimal mix:', p)
print('Defender optimal mix:', q)
print('Game value:', v)
print('Expected utility:', mpm.expected_utility(p, q, reduced.payoff))


Attacker optimal mix: [0.5 0.5]
Defender optimal mix: [0.5 0.5 0. ]
Game value: 0.5
Expected utility: 0.5


## 5) Coordinated-attack extension check
This final block checks the optional coordinated attack extension (`Δ4 = Δ1 + Δ2`).

**What it demonstrates:** whether coordinated composition defeats all admissible defenses in this setup.

In [6]:
print('Coordinated attack Δ4 defeats all admissible defenses:', coordinated_attack_defeats_all())


Coordinated attack Δ4 defeats all admissible defenses: True


## Takeaways
- This notebook is a faithful numerical replication workflow for the finite-action example.
- Agreement with expected values is an implementation sanity check, not a general proof.